# Experiment: English Transcript Role Alignment Pipeline

Objective:
- Build an English Whisper-like pipeline that stays 1:1 with the translated `role_labeled/*.json` segments.
- Reuse the original English transcript words and timestamps from `transcribed/*.json`.
- Validate the method on `305.json` and then audit the full paired corpus.


In [2]:
from __future__ import annotations

# Use the repo .venv kernel for the full openwillis stack.
import copy
import json
import re
import sys
from collections import Counter
from pathlib import Path
from pprint import pprint

import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(items, **kwargs):
        return items

try:
    from IPython.display import display
except Exception:
    def display(obj):
        print(obj)

PROJECT_ROOT = Path("/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest")
TRANSCRIBED_DIR = Path("/Users/pelmeshek1706/Downloads/Telegram Desktop/all_json_canonical")
ROLE_LABELED_DIR = Path("/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled")

FILE_ID = "305"
TRANSCRIBED_JSON = TRANSCRIBED_DIR / f"{FILE_ID}.json"
ROLE_LABELED_JSON = ROLE_LABELED_DIR / f"{FILE_ID}.json"
OUTPUT_JSON = PROJECT_ROOT / "tmp" / f"role_labeled_{FILE_ID}_source_en_whisper_like.json"
INPUT_DIR = ROLE_LABELED_DIR
OUTPUT_DIR = PROJECT_ROOT / "tmp" / "role_labeled_whisper_like_stub_batch_ukr_26-03-2026"
ENGLISH_OUTPUT_DIR = PROJECT_ROOT / "tmp" / "role_labeled_source_en_whisper_like_batch"
OWS_SRC = PROJECT_ROOT / "openwillis" / "openwillis-speech" / "src"

TARGET_SPEAKERS = {
    "participant": "participant",
    "interviewer": "interviewer",
}
TURN_MODES = ["speaker", "segment"]
DROP_UNKNOWN = True
KNOWN_ROLES = {"participant", "interviewer", "unknown", "mixed"}

if str(OWS_SRC) not in sys.path:
    sys.path.insert(0, str(OWS_SRC))

TRANSCRIBED_JSON, ROLE_LABELED_JSON, OUTPUT_JSON, INPUT_DIR, OUTPUT_DIR, ENGLISH_OUTPUT_DIR


/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


(PosixPath('/Users/pelmeshek1706/Downloads/Telegram Desktop/all_json_canonical/305.json'),
 PosixPath('/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled/305.json'),
 PosixPath('/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_305_source_en_whisper_like.json'),
 PosixPath('/Users/pelmeshek1706/Downloads/Telegram Desktop/translated_uk_gpt54_flex_full 4/role_labeled'),
 PosixPath('/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026'),
 PosixPath('/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_source_en_whisper_like_batch'))

## Plan

- Treat translated `segments` as the canonical segmentation, because they already encode the final role-labeled split.
- Pull English text and exact word ranges from the original `transcribed/*.json` via `source_turn_idx`, `source_word_start_idx`, and `source_word_end_idx`.
- Drop `unknown` segments only after that alignment, so the kept English segment count matches the kept translated segment count exactly.
- Run corpus QA to confirm that this segment-level English pipeline is valid for all paired files, not only for `305.json`.


In [3]:
SPACE_RE = re.compile(r"\s+")
TOKEN_RE = re.compile(r"\S+")


def collapse_ws(value) -> str:
    if value is None:
        return ""
    return SPACE_RE.sub(" ", str(value)).strip()


def normalize_role(value) -> str:
    role = collapse_ws(value) or "unknown"
    return role if role in KNOWN_ROLES else "unknown"


def words_to_text(words: list[dict]) -> str:
    return collapse_ws(" ".join(collapse_ws(word.get("word")) for word in words if collapse_ws(word.get("word"))))


def build_word_stub(text: str, start: float, end: float, speaker: str, score: float = 1.0) -> list[dict]:
    tokens = TOKEN_RE.findall(collapse_ws(text))
    if not tokens:
        return []
    start = float(start or 0.0)
    end = float(end if end is not None else start)
    if end < start:
        end = start
    duration = max(end - start, 0.0)
    step = duration / max(len(tokens), 1)
    words = []
    for idx, token in enumerate(tokens):
        w_start = start + idx * step
        w_end = end if idx == len(tokens) - 1 else start + (idx + 1) * step
        words.append({
            "word": token,
            "start": round(w_start, 3),
            "end": round(max(w_end, w_start), 3),
            "score": score,
            "speaker": speaker,
        })
    return words


def describe_payload(data: dict) -> dict:
    segments = data.get("segments", [])
    word_segments = data.get("word_segments", [])
    return {
        "top_level_keys": list(data.keys()),
        "n_segments": len(segments),
        "n_word_segments": len(word_segments),
        "segment_roles": Counter(seg.get("role") for seg in segments if "role" in seg),
        "segment_speakers": Counter(seg.get("speaker") for seg in segments if seg.get("speaker") is not None),
        "segments_with_words": sum(1 for seg in segments if seg.get("words")),
    }


def validate_translated_segments_against_source(transcribed: dict, role_labeled: dict) -> list[dict]:
    source_segments = transcribed.get("segments", [])
    rows = []
    for seg in role_labeled.get("segments", []):
        turn_idx = seg.get("source_turn_idx")
        role = normalize_role(seg.get("role"))
        ok_turn = isinstance(turn_idx, int) and 0 <= turn_idx < len(source_segments)
        source_turn = source_segments[turn_idx] if ok_turn else None
        start_idx = seg.get("source_word_start_idx")
        end_idx = seg.get("source_word_end_idx")
        source_words = source_turn.get("words", []) if source_turn else []
        ok_word_span = (
            ok_turn
            and isinstance(start_idx, int)
            and isinstance(end_idx, int)
            and 0 <= start_idx <= end_idx < len(source_words)
        )
        extracted_words = [copy.deepcopy(word) for word in source_words[start_idx:end_idx + 1]] if ok_word_span else []
        extracted_text = words_to_text(extracted_words)
        expected_text = collapse_ws(seg.get("source_text_en"))
        rows.append({
            "segment_id": seg.get("id"),
            "role": role,
            "keep_after_drop_unknown": role != "unknown",
            "source_turn_idx": turn_idx,
            "has_source_turn": ok_turn,
            "has_word_span": ok_word_span,
            "source_text_match": collapse_ws(extracted_text).lower() == expected_text.lower(),
            "source_turn_speaker_original": source_turn.get("speaker") if source_turn else None,
            "expected_text": expected_text,
            "extracted_text": extracted_text,
        })
    return rows


def convert_translated_segments_to_english_whisper_like(transcribed: dict, role_labeled: dict, drop_unknown: bool = True) -> tuple[dict, list[dict]]:
    source = copy.deepcopy(role_labeled)
    source_segments = transcribed.get("segments", [])
    validation_rows = validate_translated_segments_against_source(transcribed, role_labeled)
    validation_by_id = {row["segment_id"]: row for row in validation_rows}

    out_segments = []
    out_word_segments = []
    out_turn_decisions = []
    fallback_word_stub_count = 0

    for seg in source.get("segments", []):
        role = normalize_role(seg.get("role"))
        if drop_unknown and role == "unknown":
            continue

        turn_idx = seg.get("source_turn_idx")
        source_turn = source_segments[turn_idx]
        start_idx = seg.get("source_word_start_idx")
        end_idx = seg.get("source_word_end_idx")
        source_words = source_turn.get("words", [])

        extracted_words = []
        if (
            isinstance(start_idx, int)
            and isinstance(end_idx, int)
            and 0 <= start_idx <= end_idx < len(source_words)
        ):
            extracted_words = [copy.deepcopy(word) for word in source_words[start_idx:end_idx + 1]]

        if extracted_words:
            aligned_words = []
            for word in extracted_words:
                word["speaker_original"] = word.get("speaker")
                word["speaker"] = role
                aligned_words.append(word)
        else:
            fallback_word_stub_count += 1
            aligned_words = build_word_stub(seg.get("source_text_en", ""), seg.get("start", 0.0), seg.get("end", 0.0), role)
            for word in aligned_words:
                word["speaker_original"] = None

        new_seg = copy.deepcopy(seg)
        new_seg["translated_text"] = seg.get("text")
        new_seg["text"] = seg.get("source_text_en", "")
        new_seg["speaker"] = role
        new_seg["speaker_original"] = source_turn.get("speaker")
        new_seg["role"] = role
        new_seg["words"] = aligned_words
        new_seg["english_alignment"] = {
            "source": "translated_segments",
            "source_turn_idx": turn_idx,
            "source_word_start_idx": start_idx,
            "source_word_end_idx": end_idx,
            "source_turn_speaker_original": source_turn.get("speaker"),
            "source_text_match": validation_by_id[new_seg.get("id")]["source_text_match"],
        }
        out_segments.append(new_seg)
        out_word_segments.extend(copy.deepcopy(aligned_words))

    for turn in source.get("turn_decisions", []):
        role = normalize_role(turn.get("role"))
        if drop_unknown and role == "unknown":
            continue
        new_turn = copy.deepcopy(turn)
        new_turn["speaker"] = role
        out_turn_decisions.append(new_turn)

    result = {
        key: value
        for key, value in source.items()
        if key not in {"segments", "word_segments", "text"}
    }
    result["segments"] = out_segments
    result["word_segments"] = out_word_segments
    result["text"] = " ".join(collapse_ws(seg.get("text")) for seg in out_segments if collapse_ws(seg.get("text")))
    result["turn_decisions"] = out_turn_decisions
    result["source_text_en"] = result["text"]
    result["english_pipeline_meta"] = {
        "file_id": FILE_ID,
        "transcribed_file": str(TRANSCRIBED_JSON),
        "role_labeled_file": str(ROLE_LABELED_JSON),
        "drop_unknown": drop_unknown,
        "fallback_word_stub_count": fallback_word_stub_count,
        "translated_nonunknown_segments": sum(1 for seg in source.get("segments", []) if normalize_role(seg.get("role")) != "unknown"),
        "converted_segments": len(out_segments),
        "converted_word_segments": len(out_word_segments),
    }
    return result, validation_rows


def convert_transcribed_and_role_labeled_directories_to_english_whisper_like(
    transcribed_dir: str | Path,
    role_labeled_dir: str | Path,
    output_dir: str | Path,
    drop_unknown: bool = True,
) -> list[dict]:
    """Convert every common file id from `transcribed_dir` and `role_labeled_dir` into `output_dir`."""
    transcribed_dir = Path(transcribed_dir)
    role_labeled_dir = Path(role_labeled_dir)
    output_dir = Path(output_dir)

    if not transcribed_dir.exists():
        raise FileNotFoundError(f"Transcribed directory not found: {transcribed_dir}")
    if not role_labeled_dir.exists():
        raise FileNotFoundError(f"Role-labeled directory not found: {role_labeled_dir}")
    if not transcribed_dir.is_dir():
        raise NotADirectoryError(f"Transcribed path is not a directory: {transcribed_dir}")
    if not role_labeled_dir.is_dir():
        raise NotADirectoryError(f"Role-labeled path is not a directory: {role_labeled_dir}")

    common_ids = sorted({path.stem for path in transcribed_dir.glob("*.json")} & {path.stem for path in role_labeled_dir.glob("*.json")})
    output_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []

    for file_id in tqdm(common_ids, desc="source_en -> whisper_like", unit="file"):
        transcribed = json.loads((transcribed_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        role_labeled = json.loads((role_labeled_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        converted, validation_rows = convert_translated_segments_to_english_whisper_like(
            transcribed,
            role_labeled,
            drop_unknown=drop_unknown,
        )
        output_path = output_dir / f"{file_id}.json"
        output_path.write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding="utf-8")
        summary_rows.append({
            "file_id": file_id,
            "translated_nonunknown_segments": sum(1 for row in validation_rows if row["keep_after_drop_unknown"]),
            "output_segments": len(converted.get("segments", [])),
            "output_word_segments": len(converted.get("word_segments", [])),
            "output_path": str(output_path),
        })

    return summary_rows


def convert_role_labeled_to_whisper_like(data: dict, drop_unknown: bool = True) -> dict:
    source = copy.deepcopy(data)
    new_segments = []
    new_word_segments = []
    new_turn_decisions = []

    for seg in source.get("segments", []):
        role = seg.get("role")
        if drop_unknown and role == "unknown":
            continue

        speaker = role or "unknown"
        new_seg = copy.deepcopy(seg)
        new_seg["speaker"] = speaker
        new_seg["words"] = build_word_stub(
            text=new_seg.get("text", ""),
            start=new_seg.get("start", 0.0),
            end=new_seg.get("end", 0.0),
            speaker=speaker,
        )
        new_segments.append(new_seg)
        new_word_segments.extend(copy.deepcopy(new_seg["words"]))

    for turn in source.get("turn_decisions", []):
        role = turn.get("role")
        if drop_unknown and role == "unknown":
            continue
        new_turn = copy.deepcopy(turn)
        new_turn["speaker"] = role or "unknown"
        new_turn_decisions.append(new_turn)

    result = {
        key: value
        for key, value in source.items()
        if key not in {"segments", "word_segments", "turn_decisions", "text", "source_text_en"}
    }
    result["segments"] = new_segments
    result["word_segments"] = new_word_segments
    result["text"] = " ".join(seg.get("text", "") for seg in new_segments).strip()
    result["source_text_en"] = " ".join(seg.get("source_text_en", "") for seg in new_segments).strip()
    result["turn_decisions"] = new_turn_decisions
    return result


def convert_role_labeled_directory_to_whisper_like(
    input_dir: str | Path,
    output_dir: str | Path,
    drop_unknown: bool = True,
) -> list[dict]:
    """Convert every `*.json` in `input_dir` and write it to `output_dir` with the same file name."""
    input_dir = Path(input_dir)
    output_dir = Path(output_dir)

    if not input_dir.exists():
        raise FileNotFoundError(f"Input directory not found: {input_dir}")
    if not input_dir.is_dir():
        raise NotADirectoryError(f"Input path is not a directory: {input_dir}")

    output_dir.mkdir(parents=True, exist_ok=True)
    summary_rows = []
    input_paths = sorted(input_dir.glob("*.json"))

    for input_path in tqdm(input_paths, desc="role_labeled -> whisper_like", unit="file"):
        raw = json.loads(input_path.read_text(encoding="utf-8"))
        converted = convert_role_labeled_to_whisper_like(raw, drop_unknown=drop_unknown)
        output_path = output_dir / input_path.name
        output_path.write_text(json.dumps(converted, ensure_ascii=False, indent=2), encoding="utf-8")
        summary_rows.append({
            "file_name": input_path.name,
            "input_segments": len(raw.get("segments", [])),
            "output_segments": len(converted.get("segments", [])),
            "output_word_segments": len(converted.get("word_segments", [])),
            "output_path": str(output_path),
        })

    return summary_rows


def audit_pair(file_id: str, transcribed: dict, role_labeled: dict) -> dict:
    converted, validation_rows = convert_translated_segments_to_english_whisper_like(transcribed, role_labeled, drop_unknown=DROP_UNKNOWN)
    kept_rows = [row for row in validation_rows if row["keep_after_drop_unknown"]]
    return {
        "file_id": file_id,
        "translated_segments": len(role_labeled.get("segments", [])),
        "translated_nonunknown_segments": sum(row["keep_after_drop_unknown"] for row in validation_rows),
        "converted_segments": len(converted.get("segments", [])),
        "converted_words": len(converted.get("word_segments", [])),
        "missing_source_turns": sum(not row["has_source_turn"] for row in kept_rows),
        "missing_word_spans": sum(not row["has_word_span"] for row in kept_rows),
        "source_text_mismatches": sum(not row["source_text_match"] for row in kept_rows),
        "fallback_word_stub_count": converted["english_pipeline_meta"]["fallback_word_stub_count"],
        "roles": dict(Counter(seg.get("role") for seg in converted.get("segments", []))),
    }


def collect_corpus_audit(transcribed_dir: Path, role_labeled_dir: Path) -> pd.DataFrame:
    common_ids = sorted({path.stem for path in transcribed_dir.glob("*.json")} & {path.stem for path in role_labeled_dir.glob("*.json")})
    rows = []
    for file_id in common_ids:
        transcribed = json.loads((transcribed_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        role_labeled = json.loads((role_labeled_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        rows.append(audit_pair(file_id, transcribed, role_labeled))
    return pd.DataFrame(rows)


In [5]:
raw_transcribed = json.loads(TRANSCRIBED_JSON.read_text(encoding="utf-8"))
raw_role_labeled = json.loads(ROLE_LABELED_JSON.read_text(encoding="utf-8"))

converted_en, validation_rows = convert_translated_segments_to_english_whisper_like(
    raw_transcribed,
    raw_role_labeled,
    drop_unknown=DROP_UNKNOWN,
)
validation_df = pd.DataFrame(validation_rows)
pair_summary = audit_pair(FILE_ID, raw_transcribed, raw_role_labeled)

print("Original English transcribed payload:")
pprint(describe_payload(raw_transcribed))

print("\nTranslated role-labeled payload:")
pprint(describe_payload(raw_role_labeled))

print("\nEnglish payload built from translated segments:")
pprint(describe_payload(converted_en))

print("\nPair audit summary:")
pprint(pair_summary)

print("\nValidation sample:")
display(validation_df.head(12))

print("\nSample converted segment (without full words payload):")
sample_segment = {k: v for k, v in converted_en["segments"][0].items() if k != "words"}
pprint(sample_segment)
print("sample words:", converted_en["segments"][0]["words"][:5])


Original English transcribed payload:
{'n_segments': 323,
 'n_word_segments': 3505,
 'segment_roles': Counter(),
 'segment_speakers': Counter({'SPEAKER_02': 225,
                              'SPEAKER_01': 88,
                              'SPEAKER_00': 10}),
 'segments_with_words': 323,
 'top_level_keys': ['segments', 'word_segments']}

Translated role-labeled payload:
{'n_segments': 331,
 'n_word_segments': 0,
 'segment_roles': Counter({'participant': 228,
                           'interviewer': 93,
                           'unknown': 10}),
 'segment_speakers': Counter(),
 'segments_with_words': 0,
 'top_level_keys': ['segments',
                    'word_segments',
                    'text',
                    'source_text_en',
                    'translation_meta',
                    'source_cleanup_meta',
                    'turn_decisions']}

English payload built from translated segments:
{'n_segments': 321,
 'n_word_segments': 3406,
 'segment_roles': Counter({'particip

,segment_id,role,keep_after_drop_unknown,source_turn_idx,has_source_turn,has_word_span,source_text_match,source_turn_speaker_original,expected_text,extracted_text
0,0,unknown,False,0,True,True,True,SPEAKER_00,"Okay, looks good.","Okay, looks good."
1,1,unknown,False,1,True,True,True,SPEAKER_00,So if we can just move around a little bit to ...,So if we can just move around a little bit to ...
2,2,unknown,False,2,True,True,True,SPEAKER_00,We'll just move around.,We'll just move around.
3,3,unknown,False,3,True,True,True,SPEAKER_00,There it goes.,There it goes.
4,4,unknown,False,4,True,True,True,SPEAKER_00,"Okay, perfect.","Okay, perfect."
5,5,unknown,False,5,True,True,True,SPEAKER_00,So I'm going to go ahead and start up the virt...,So I'm going to go ahead and start up the virt...
6,6,unknown,False,6,True,True,True,SPEAKER_00,"When it's over, will you just go ahead and rin...","When it's over, will you just go ahead and rin..."
7,7,participant,True,7,True,True,True,SPEAKER_02,"Okay, great.","Okay, great."
8,8,participant,True,8,True,True,True,SPEAKER_02,Will it let me know that I'm done?,Will it let me know that I'm done?
9,9,unknown,False,9,True,True,True,SPEAKER_00,Yes.,Yes.



Sample converted segment (without full words payload):
{'decision_source': 'model_turn_pass',
 'end': 20.752,
 'english_alignment': {'source': 'translated_segments',
                       'source_text_match': True,
                       'source_turn_idx': 7,
                       'source_turn_speaker_original': 'SPEAKER_02',
                       'source_word_end_idx': 1,
                       'source_word_start_idx': 0},
 'id': 7,
 'needs_review': False,
 'role': 'participant',
 'source_text_en': 'Okay, great.',
 'source_turn_idx': 7,
 'source_word_end_idx': 1,
 'source_word_start_idx': 0,
 'speaker': 'participant',
 'speaker_original': 'SPEAKER_02',
 'start': 20.071,
 'text': 'Okay, great.',
 'translated': True,
 'translated_text': 'Добре, чудово.',
 'translation_confidence': 'high',
 'translation_needs_review': False,
 'translation_qa_flags': [],
 'turn_role_decision': 'participant'}
sample words: [{'word': 'Okay,', 'start': 20.071, 'end': 20.331, 'score': 0.155, 'speaker': 'p

In [ ]:
OUTPUT_JSON.parent.mkdir(parents=True, exist_ok=True)
OUTPUT_JSON.write_text(json.dumps(converted_en, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"Wrote: {OUTPUT_JSON}")
print(f"Segments kept: {len(converted_en['segments'])}")
print(f"Word segments kept: {len(converted_en['word_segments'])}")
print(f"Turn decisions kept: {len(converted_en['turn_decisions'])}")


In [ ]:
batch_summary = convert_transcribed_and_role_labeled_directories_to_english_whisper_like(
    transcribed_dir=TRANSCRIBED_DIR,
    role_labeled_dir=ROLE_LABELED_DIR,
    output_dir=ENGLISH_OUTPUT_DIR,
    drop_unknown=DROP_UNKNOWN,
)
print(f"Files written: {len(batch_summary)}")
display(pd.DataFrame(batch_summary).head(3))


In [ ]:
batch_summary = convert_role_labeled_directory_to_whisper_like(
    input_dir=INPUT_DIR,
    output_dir=OUTPUT_DIR,
    drop_unknown=DROP_UNKNOWN,
)
print(f"Files written: {len(batch_summary)}")
pprint(batch_summary[:3])


In [6]:
DATASET_ENG_DIR = Path("/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_eng_26-03-2026")
DATASET_UKR_DIR = Path("/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_ukr_26-03-2026")
DATASET_OUTPUT_CSV = PROJECT_ROOT / "tmp" / "eng_ukr_segment_dataset_26-03-2026.csv"


def get_speaker_type(seg: dict) -> str:
    for key in ("speaker", "role", "speaker_original"):
        value = collapse_ws(seg.get(key))
        if value:
            return value
    return "unknown"


def same_time(left, right, tol: float = 1e-6) -> bool:
    if left is None or right is None:
        return left == right
    return abs(float(left) - float(right)) <= tol


def build_eng_ukr_segment_dataset(eng_dir: str | Path, ukr_dir: str | Path) -> pd.DataFrame:
    eng_dir = Path(eng_dir)
    ukr_dir = Path(ukr_dir)
    common_ids = sorted({path.stem for path in eng_dir.glob("*.json")} & {path.stem for path in ukr_dir.glob("*.json")})
    if not common_ids:
        raise FileNotFoundError("No common JSON files found between English and Ukrainian directories.")

    rows = []
    for file_id in tqdm(common_ids, desc="eng+ukr -> dataset", unit="file"):
        eng = json.loads((eng_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        ukr = json.loads((ukr_dir / f"{file_id}.json").read_text(encoding="utf-8"))
        eng_segments = eng.get("segments", [])
        ukr_segments = ukr.get("segments", [])

        if len(eng_segments) != len(ukr_segments):
            raise ValueError(
                f"Segment count mismatch for {file_id}: {len(eng_segments)} != {len(ukr_segments)}"
            )

        for idx, (eng_seg, ukr_seg) in enumerate(zip(eng_segments, ukr_segments)):
            eng_speaker = get_speaker_type(eng_seg)
            ukr_speaker = get_speaker_type(ukr_seg)
            ids_match = eng_seg.get("id") == ukr_seg.get("id")
            speakers_match = eng_speaker == ukr_speaker
            starts_match = same_time(eng_seg.get("start"), ukr_seg.get("start"))
            ends_match = same_time(eng_seg.get("end"), ukr_seg.get("end"))
            if not (ids_match and speakers_match and starts_match and ends_match):
                raise ValueError(
                    "Alignment mismatch for "
                    f"{file_id} at segment #{idx}: "
                    f"id_match={ids_match}, speaker_match={speakers_match}, "
                    f"start_match={starts_match}, end_match={ends_match}"
                )

            rows.append({
                "file_name": file_id,
                "speaker_type": eng_speaker,
                "lang_ukr": collapse_ws(ukr_seg.get("text")),
                "lang_eng": collapse_ws(eng_seg.get("text")),
            })

    return pd.DataFrame(rows, columns=["file_name", "speaker_type", "lang_ukr", "lang_eng"])


eng_ukr_segment_dataset = build_eng_ukr_segment_dataset(DATASET_ENG_DIR, DATASET_UKR_DIR)
DATASET_OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
eng_ukr_segment_dataset.to_csv(DATASET_OUTPUT_CSV, index=False)

print(f"Wrote dataset: {DATASET_OUTPUT_CSV}")
print(f"Rows: {len(eng_ukr_segment_dataset)}")
print(f"Files: {eng_ukr_segment_dataset['file_name'].nunique()}")
display(eng_ukr_segment_dataset.head(10))
display(
    eng_ukr_segment_dataset["speaker_type"]
    .value_counts()
    .rename_axis("speaker_type")
    .reset_index(name="rows")
)


eng+ukr -> dataset: 100%|██████████| 275/275 [00:02<00:00, 110.14file/s]


Wrote dataset: /Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/eng_ukr_segment_dataset_26-03-2026.csv
Rows: 44309
Files: 275


,file_name,speaker_type,lang_ukr,lang_eng
0,300,participant,Добре.,OK.
1,300,interviewer,"Привіт, я Еллі.","Hi, I'm Ellie."
2,300,interviewer,"Дякую, що прийшли сьогодні.",Thanks for coming in today.
3,300,interviewer,"Мене створили, щоб розмовляти з людьми в безпе...",I was created to talk to people in a safe and ...
4,300,interviewer,Сприймайте мене як друга.,Think of me as a friend.
5,300,interviewer,Я не засуджую.,I don't judge.
6,300,interviewer,Не можу.,I can.
7,300,interviewer,Я комп'ютер.,I'm a computer.
8,300,interviewer,"Я тут, щоб дізнаватися про людей, і мені б дуж...",I'm here to learn about people and would love ...
9,300,interviewer,"Я поставлю кілька запитань, щоб ми могли почати.",I'll ask a few questions to get us started.


,speaker_type,rows
0,participant,31500
1,interviewer,12809


## Test `openwillis.speech`

The fast structural check below validates that the English segment-aligned object is accepted by `filter_whisper()` in both `speaker` and `segment` modes.


In [ ]:
import openwillis.speech as ows
from openwillis.speech.speech_attribute import filter_whisper, get_config
from openwillis.speech.util.speech import coherence

coherence.COHERENCE_BACKEND = "gemma"
measures = get_config(str((OWS_SRC / "openwillis" / "speech" / "speech_attribute.py").resolve()), "text.json")

utterance_views = {}
utterance_summary_rows = []
for turn_mode in TURN_MODES:
    _, utterances_i = filter_whisper(converted_en, measures, whisper_turn_mode=turn_mode)
    utterance_views[turn_mode] = utterances_i[[
        measures["speaker_label"],
        measures["utterance_ids"],
        measures["utterance_text"],
    ]].copy()
    utterance_summary_rows.append({
        "turn_mode": turn_mode,
        "num_utterances": len(utterances_i),
        "speaker_counts": utterances_i[measures["speaker_label"]].value_counts(dropna=False).to_dict(),
    })

utterance_summary = pd.DataFrame(utterance_summary_rows)
display(utterance_summary)
display(utterance_views["speaker"].head(12))
display(utterance_views["segment"].head(12))


### Coherence Features

This step is heavier than the structural validation above.

- Run it inside the repo kernel at `/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/bin/python`.
- On the first run, `openwillis.speech` may download/load `embeddinggemma` and `gemma-3-270m`, so expect a noticeable startup delay.


In [4]:
comparison_rows = []
feature_outputs = {}
for speaker_name, label in TARGET_SPEAKERS.items():
    for turn_mode in TURN_MODES:
        words_i, turns_i, summary_i = ows.speech_characteristics(
            json_conf=converted_en,
            option="coherence",
            language="en",
            speaker_label=label,
            min_coherence_turn_length=2,
            whisper_turn_mode=turn_mode,
        )
        feature_outputs[(speaker_name, turn_mode)] = {
            "words": words_i,
            "turns": turns_i,
            "summary": summary_i,
        }
        summary_row = summary_i.iloc[0] if not summary_i.empty else {}
        comparison_rows.append({
            "speaker_name": speaker_name,
            "speaker_label": label,
            "turn_mode": turn_mode,
            "words_rows": len(words_i),
            "turn_rows": len(turns_i),
            "speaker_percentage": summary_row.get("speaker_percentage"),
            "speech_minutes": summary_row.get("speech_length_minutes"),
            "speech_words": summary_row.get("speech_length_words"),
            "mean_turn_words": summary_row.get("mean_turn_length_words"),
            "num_turns": summary_row.get("num_turns"),
        })

comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
display(feature_outputs[("participant", "segment")]["turns"].head(5))
display(feature_outputs[("interviewer", "segment")]["turns"].head(5))


NameError: name 'ows' is not defined

## Corpus QA

This checks whether the segment-level English pipeline stays aligned with the translated segment payload across the whole paired dataset.


In [ ]:
corpus_df = collect_corpus_audit(TRANSCRIBED_DIR, ROLE_LABELED_DIR).sort_values("file_id").reset_index(drop=True)

corpus_overview = {
    "paired_files": int(len(corpus_df)),
    "files_with_segment_count_mismatch": int((corpus_df["translated_nonunknown_segments"] != corpus_df["converted_segments"]).sum()),
    "total_missing_source_turns": int(corpus_df["missing_source_turns"].sum()),
    "total_missing_word_spans": int(corpus_df["missing_word_spans"].sum()),
    "total_source_text_mismatches": int(corpus_df["source_text_mismatches"].sum()),
    "total_fallback_word_stubs": int(corpus_df["fallback_word_stub_count"].sum()),
}
pprint(corpus_overview)

print("\nRows for 305 / 306 / 307:")
display(corpus_df[corpus_df["file_id"].isin(["305", "306", "307"])])

print("\nCorpus head:")
display(corpus_df.head(12))


## Results

- The correct English pipeline for this dataset is segment-based, not turn-based.
- It keeps the translated `role_labeled` segmentation 1:1 and swaps in original English text plus original English word timestamps.
- On the current paired corpus, the pipeline yields zero segment-count mismatches, zero missing source turns, zero missing word spans, zero source-text mismatches, and zero fallback word stubs.


# check translation quality

In [12]:
import re
import numpy as np
import pandas as pd
from lexicalrichness import LexicalRichness
from comet import download_model, load_from_checkpoint
import nltk
from nltk.translate.meteor_score import meteor_score

nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

EN_COL = "lang_eng"
UA_COL = "lang_ukr"
UA_REF_COL = "ua_ref"
EN_BACK_COL = "Text_eng_back"

MIN_TOKENS = 5
MATTR_WINDOW = 50

TOKEN_RE = re.compile(r"[^\W\d_]+", re.UNICODE)

def tokenize(text: str) -> list[str]:
    if not isinstance(text, str):
        return []
    return TOKEN_RE.findall(text.lower())

# ✅ НЕ делаем astype(str), чтобы NaN не стал "nan"
def token_len_series(s: pd.Series) -> pd.Series:
    return s.map(lambda x: len(tokenize(x)) if isinstance(x, str) else np.nan)

def safe_mean_std(x) -> tuple[float, float]:
    # на случай дублей колонок
    if isinstance(x, pd.DataFrame):
        x = x.iloc[:, -1]
    x = pd.to_numeric(x, errors="coerce").dropna()
    if x.empty:
        return float("nan"), float("nan")
    return float(x.mean()), float(x.std(ddof=1))

# --- COMET QE ---
# ✅ грузим модель 1 раз, а не внутри каждого вызова
MODEL_NAME = "Unbabel/wmt22-cometkiwi-da"
model_path = download_model(MODEL_NAME)
comet_model = load_from_checkpoint(model_path)

def compute_comet_qe(df: pd.DataFrame, src_col: str, mt_col: str, gpus: int = 0) -> tuple[pd.Series, float]:
    data = [{"src": s, "mt": t} for s, t in zip(df[src_col], df[mt_col])]
    out = comet_model.predict(data, batch_size=8, gpus=gpus)
    return pd.Series(out.scores, index=df.index), float(out.system_score)

# --- METEOR ---
def mean_meteor(hyp: list[str], ref: list[str]) -> float:
    scores = []
    for h, r in zip(hyp, ref):
        h_tok = tokenize(h)
        r_tok = tokenize(r)
        scores.append(meteor_score([r_tok], h_tok))
    return float(np.mean(scores)) if scores else float("nan")

# --- Lexical diversity ---
# ✅ КРИТИЧНО: возвращаем DataFrame с index=series.index, чтобы ничего не "переехало"
def lexical_stats(series: pd.Series, mattr_window: int = 50) -> pd.DataFrame:
    rows = []
    for text in series:
        toks = tokenize(text) if isinstance(text, str) else []
        if not toks:
            rows.append({"ttr": np.nan, "mattr": np.nan, "n_tokens": np.nan})
            continue

        lex = LexicalRichness(" ".join(toks))
        rows.append({
            "ttr": float(lex.ttr),
            "mattr": float(lex.mattr(window_size=min(mattr_window, max(5, len(toks))))),
            # ✅ n_tokens считаем сами, не через lex.words (так стабильнее)
            "n_tokens": float(len(toks)),
        })
    return pd.DataFrame(rows, index=series.index)


Fetching 5 files: 100%|██████████| 5/5 [00:00<00:00, 42027.09it/s]
Lightning automatically upgraded your loaded checkpoint from v1.8.2 to v2.5.1. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint ../../../../../../.cache/huggingface/hub/models--Unbabel--wmt22-cometkiwi-da/snapshots/1ad785194e391eebc6c53e2d0776cada8f83179a/checkpoints/model.ckpt`
Encoder model frozen.
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/pytorch_lightning/core/saving.py:195: Found keys that are not in the model state dict but in the checkpoint: ['encoder.model.embeddings.position_ids']


In [11]:
eng_ukr_segment_dataset.head(5)

,file_name,speaker_type,lang_ukr,lang_eng
0,300,participant,Добре.,OK.
1,300,interviewer,"Привіт, я Еллі.","Hi, I'm Ellie."
2,300,interviewer,"Дякую, що прийшли сьогодні.",Thanks for coming in today.
3,300,interviewer,"Мене створили, щоб розмовляти з людьми в безпе...",I was created to talk to people in a safe and ...
4,300,interviewer,Сприймайте мене як друга.,Think of me as a friend.


In [13]:

# ----------------- RUN -----------------
# df = pd.read_csv("/Users/pelmeshek1706/Desktop/projects/dcapwoz_all_plus_ukr_eng_back.csv")
# df = pd.read_csv("df_retranslated_20-12-25.csv")
df = eng_ukr_segment_dataset.copy()
missing = [c for c in (EN_COL, UA_COL) if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns: {missing}. Available: {list(df.columns)}")

# 1) COMET QE
df["comet_qe"], comet_system = compute_comet_qe(df, src_col=EN_COL, mt_col=UA_COL, gpus=1)
print("COMET-QE system score (EN->UA):", comet_system)

You are using the plain ModelCheckpoint callback. Consider using LitModelCheckpoint which with seamless uploading to Model registry.
GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
Predicting DataLoader 0: 100%|██████████| 5539/5539 [06:02<00:00, 15.29it/s]


COMET-QE system score (EN->UA): 0.8337267154000876


In [14]:
# 2) Sentence length
en_len_col = f"token_len_{EN_COL}"
ua_len_col = f"token_len_{UA_COL}"
df[en_len_col] = token_len_series(df[EN_COL])
df[ua_len_col] = token_len_series(df[UA_COL])

# en_mean, en_std = safe_mean_std(df[en_len_col])
# ua_mean, ua_std = safe_mean_std(df[ua_len_col])
# print(f"Sentence length (tokens) EN mean±std: {en_mean:.3f} ± {en_std:.3f}")
# print(f"Sentence length (tokens) UA mean±std: {ua_mean:.3f} ± {ua_std:.3f}")

TRIM_Q = 0.995  # отсекаем top 5%

en_len = df[en_len_col]
ua_len = df[ua_len_col]

en_thr = en_len.quantile(TRIM_Q)
ua_thr = ua_len.quantile(TRIM_Q)

mask_en_trim = en_len.notna() & (en_len <= en_thr)
mask_ua_trim = ua_len.notna() & (ua_len <= ua_thr)

# если хочешь общий фильтр строк (и EN и UA без выбросов):
mask_trim_both = mask_en_trim & mask_ua_trim
en_mean_trim, en_std_trim = safe_mean_std(en_len[mask_en_trim])
ua_mean_trim, ua_std_trim = safe_mean_std(ua_len[mask_ua_trim])

print("EN trimmed 99.5% mean±std:", en_mean_trim, en_std_trim,)# "thr:", en_thr)
print("UA trimmed 99.5% mean±std:", ua_mean_trim, ua_std_trim,)# "thr:", ua_thr)

df["len_ratio_ua_en"] = df[ua_len_col] / df[en_len_col].replace(0, np.nan)
ratio_mean, ratio_std = safe_mean_std(df["len_ratio_ua_en"])
print(f"Length ratio UA/EN mean±std: {ratio_mean:.3f} ± {ratio_std:.3f}")

# 3) Lexical diversity with proper masks (>= MIN_TOKENS AND notna)
mask_en = df[en_len_col].notna() & (df[en_len_col] >= MIN_TOKENS)
mask_ua = df[ua_len_col].notna() & (df[ua_len_col] >= MIN_TOKENS)

# ✅ перед повторным запуском убираем старые en_/ua_ колонки, иначе будут дубли
df = df.drop(columns=[c for c in df.columns if c.startswith(("en_", "ua_"))], errors="ignore")

df_lex_en = lexical_stats(df.loc[mask_en, EN_COL], mattr_window=MATTR_WINDOW).add_prefix("en_")
df_lex_ua = lexical_stats(df.loc[mask_ua, UA_COL], mattr_window=MATTR_WINDOW).add_prefix("ua_")

# ✅ join по индексу (без переездов)
df = df.join(df_lex_en, how="left").join(df_lex_ua, how="left")

en_ttr_mean, en_ttr_std = safe_mean_std(df["en_ttr"])
ua_ttr_mean, ua_ttr_std = safe_mean_std(df["ua_ttr"])
en_mattr_mean, en_mattr_std = safe_mean_std(df["en_mattr"])
ua_mattr_mean, ua_mattr_std = safe_mean_std(df["ua_mattr"])

print(f"TTR EN mean±std (filtered >= {MIN_TOKENS} toks): {en_ttr_mean:.4f} ± {en_ttr_std:.4f}")
print(f"TTR UA mean±std (filtered >= {MIN_TOKENS} toks): {ua_ttr_mean:.4f} ± {ua_ttr_std:.4f}")
print(f"MATTR EN mean±std (filtered >= {MIN_TOKENS} toks): {en_mattr_mean:.4f} ± {en_mattr_std:.4f}")
print(f"MATTR UA mean±std (filtered >= {MIN_TOKENS} toks): {ua_mattr_mean:.4f} ± {ua_mattr_std:.4f}")

# ✅ НЕ перетираем len_ratio_ua_en второй раз через en_/ua_n_tokens
df["mattr_ratio_ua_en"] = df["ua_mattr"] / df["en_mattr"].replace(0, np.nan)

# Optional: ratios for anomaly hunting
# df["len_ratio_ua_en"] = df["ua_n_tokens"] / df["en_n_tokens"].replace(0, np.nan)
# df["mattr_ratio_ua_en"] = df["ua_mattr"] / df["en_mattr"].replace(0, np.nan)
# context
# print("Lexical diversity EN MATTR mean:", float(df["en_mattr"].mean()))
# print("Lexical diversity UA MATTR mean:", float(df["ua_mattr"].mean()))
# print("Mean length ratio UA/EN:", float(df["len_ratio_ua_en"].mean()))

EN trimmed 99.5% mean±std: 9.443514549454537 9.071103508954211
UA trimmed 99.5% mean±std: 7.988138975824375 7.7281839389897975
Length ratio UA/EN mean±std: 0.873 ± 0.221
TTR EN mean±std (filtered >= 5 toks): 0.9294 ± 0.0981
TTR UA mean±std (filtered >= 5 toks): 0.9596 ± 0.0745
MATTR EN mean±std (filtered >= 5 toks): 0.9302 ± 0.0960
MATTR UA mean±std (filtered >= 5 toks): 0.9600 ± 0.0733


In [15]:
TRIM_Q = 0.995  # отсекаем top 5%

en_len = df[en_len_col]
ua_len = df[ua_len_col]

en_thr = en_len.quantile(TRIM_Q)
ua_thr = ua_len.quantile(TRIM_Q)

mask_en_trim = en_len.notna() & (en_len <= en_thr)
mask_ua_trim = ua_len.notna() & (ua_len <= ua_thr)

# если хочешь общий фильтр строк (и EN и UA без выбросов):
mask_trim_both = mask_en_trim & mask_ua_trim

# робастные mean/std после trim
en_mean_trim, en_std_trim = safe_mean_std(en_len[mask_en_trim])
ua_mean_trim, ua_std_trim = safe_mean_std(ua_len[mask_ua_trim])

print("EN trimmed 95% mean±std:", en_mean_trim, en_std_trim, "thr:", en_thr)
print("UA trimmed 95% mean±std:", ua_mean_trim, ua_std_trim, "thr:", ua_thr)
print("Dropped rows (both):", int((~mask_trim_both).sum()))


EN trimmed 95% mean±std: 9.443514549454537 9.071103508954211 thr: 68.0
UA trimmed 95% mean±std: 7.988138975824375 7.7281839389897975 thr: 57.0
Dropped rows (both): 249


In [16]:
!pip install sacrebleu


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [17]:
# pip install sacrebleu
from sacrebleu.metrics import CHRF

def nonempty_str(x):
    return isinstance(x, str) and x.strip() != ""

mask_back = df[EN_COL].map(nonempty_str) & df[EN_BACK_COL].map(nonempty_str)

refs = df.loc[mask_back, EN_COL].tolist()          # original EN
hyps = df.loc[mask_back, EN_BACK_COL].tolist()     # back-translated EN

chrfpp = CHRF(word_order=2, lowercase=False)
chrfpp_lc = CHRF(word_order=2, lowercase=True)

score_cs = chrfpp.corpus_score(hyps, [refs]).score
score_lc = chrfpp_lc.corpus_score(hyps, [refs]).score

print(f"chrF++ (EN_back vs EN, case-sensitive): {score_cs:.2f}")
print(f"chrF++ (EN_back vs EN, lowercased): {score_lc:.2f}")


KeyError: 'Text_eng_back'

In [18]:
eng_ukr_segment_dataset.head(25)

,file_name,speaker_type,lang_ukr,lang_eng
0,300,participant,Добре.,OK.
1,300,interviewer,"Привіт, я Еллі.","Hi, I'm Ellie."
2,300,interviewer,"Дякую, що прийшли сьогодні.",Thanks for coming in today.
3,300,interviewer,"Мене створили, щоб розмовляти з людьми в безпе...",I was created to talk to people in a safe and ...
4,300,interviewer,Сприймайте мене як друга.,Think of me as a friend.
5,300,interviewer,Я не засуджую.,I don't judge.
6,300,interviewer,Не можу.,I can.
7,300,interviewer,Я комп'ютер.,I'm a computer.
8,300,interviewer,"Я тут, щоб дізнаватися про людей, і мені б дуж...",I'm here to learn about people and would love ...
9,300,interviewer,"Я поставлю кілька запитань, щоб ми могли почати.",I'll ask a few questions to get us started.


# difference between whisper_turn_mode

## Segments 

In [2]:
import json

file_path = "/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/tmp/role_labeled_whisper_like_stub_batch_eng_26-03-2026/300.json"

with open(file_path, "r", encoding="utf-8") as f:
    date_eng = json.load(f)

print(date_eng)

{'source_text_en': "OK. Hi, I'm Ellie. Thanks for coming in today. I was created to talk to people in a safe and secure environment. Think of me as a friend. I don't judge. I can. I'm a computer. I'm here to learn about people and would love to learn about you. I'll ask a few questions to get us started. And please feel free to tell me anything. Your answers are totally confidential. How are you doing today? Good. Where are you from originally? Atlanta, Georgia. Really? Why did you move to LA? My parents are from here. How do you like LA? I love it. What are some things you really like about LA? I like the weather. I like the opportunities. Yes. How easy was it for you to get used to living in LA? It took a minute. Somewhat easy. What are some things you don't really like about LA? Congestion. That's it. What did you study at school? I took up business and administration. Are you still doing that? Yeah, I am. Here and there. I'm on a break right now, but I plan on going back in the nex

In [3]:
import openwillis.speech as ows
from openwillis.speech.speech_attribute import filter_whisper, get_config
from openwillis.speech.util.speech import coherence

coherence.COHERENCE_BACKEND = "gemma"

words_seg, turns_seg, summary_seg = ows.speech_characteristics(
            json_conf=date_eng,
            option="coherence",
            language="en",
            speaker_label="participant",
            min_coherence_turn_length=2,
            whisper_turn_mode='segment',
        )

/Users/pelmeshek1706/Desktop/projects/final_airest_voice/airest/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO:sentence_transformers.SentenceTransformer:Load pretrained SentenceTransformer: google/embeddinggemma-300m


Try edit function....


INFO:sentence_transformers.SentenceTransformer:14 prompts are loaded, with the keys: ['query', 'document', 'BitextMining', 'Clustering', 'Classification', 'InstructionRetrieval', 'MultilabelClassification', 'PairClassification', 'Reranking', 'Retrieval', 'Retrieval-query', 'Retrieval-document', 'STS', 'Summarization']
Batches: 100%|██████████| 3/3 [00:00<00:00,  7.05it/s]


In [5]:
import pandas as pd

pd.set_option('display.max_columns', None)

In [6]:
summary_seg

,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,2.8582,272,95.164789,140.64796,0.171854,0.229606,26.730803,0.316859,0.245741,0.4374,0.071118,0.284,0.066,0.649,0.9974,0.96,0.894909,0.792923,0.716255,0.624216,9.141274,0.279412,0.333333,28.70608,1.195718,24.702481,1.571068,0.0,0.568652,0.006675,0.66144,0.002091,0.613529,0.001278,0.565532,0.007186,0.557362,0.00823,0.576111,0.011307,0.557167,0.007614,0.558133,0.006896,0.587133,0.008856,0.602618,0.020867,0.579937,0.010857,0.570366,0.005243,66,17,0.043306,4.121212,1.864338,26.730803,NaN,NaN,NaN,NaN,0.466377,0.011043,-0.001388,1700.925378,3.398997e+07,31066.085516,2.649128e+10,5426.447874,7.917330e+08,2313.979702,6.933612e+07,0,22.092573,0.316098,22.038027


In [8]:
words_sp, turns_sp, summary_sp = ows.speech_characteristics(
            json_conf=date_eng,
            option="coherence",
            language="en",
            speaker_label="participant",
            min_coherence_turn_length=2,
            whisper_turn_mode='speaker',
        )

Try edit function....


Batches: 100%|██████████| 3/3 [00:00<00:00,  5.29it/s]


In [9]:
summary_sp

,file_length,speech_length_minutes,speech_length_words,words_per_min,syllables_per_min,mean_pre_word_pause,mean_pause_variability,speech_percentage,sentiment_pos,sentiment_neg,sentiment_neu,sentiment_overall,sentiment_vader_pos,sentiment_vader_neg,sentiment_vader_neu,sentiment_vader_overall,mattr_5,mattr_10,mattr_25,mattr_50,mattr_100,first_person_percentage,prop_verb_past,prop_function_words,first_person_sentiment_positive,first_person_sentiment_negative,first_person_sentiment_overall,word_repeat_percentage,phrase_repeat_percentage,word_coherence_mean,word_coherence_var,word_coherence_5_mean,word_coherence_5_var,word_coherence_10_mean,word_coherence_10_var,word_coherence_variability_2_mean,word_coherence_variability_2_var,word_coherence_variability_3_mean,word_coherence_variability_3_var,word_coherence_variability_4_mean,word_coherence_variability_4_var,word_coherence_variability_5_mean,word_coherence_variability_5_var,word_coherence_variability_6_mean,word_coherence_variability_6_var,word_coherence_variability_7_mean,word_coherence_variability_7_var,word_coherence_variability_8_mean,word_coherence_variability_8_var,word_coherence_variability_9_mean,word_coherence_variability_9_var,word_coherence_variability_10_mean,word_coherence_variability_10_var,num_turns,num_one_word_turns,mean_turn_length_minutes,mean_turn_length_words,mean_pre_turn_pause,speaker_percentage,first_order_sentence_tangeniality_mean,first_order_sentence_tangeniality_var,second_order_sentence_tangeniality_mean,second_order_sentence_tangeniality_var,turn_to_turn_tangeniality_mean,turn_to_turn_tangeniality_var,turn_to_turn_tangeniality_slope,semantic_perplexity_mean,semantic_perplexity_var,semantic_perplexity_5_mean,semantic_perplexity_5_var,semantic_perplexity_11_mean,semantic_perplexity_11_var,semantic_perplexity_15_mean,semantic_perplexity_15_var,num_interrupts,first_person_sentiment_positive_vader,first_person_sentiment_negative_vader,first_person_sentiment_overall_vader
0,10.692533,7.202417,619,85.943375,129.401011,0.329675,1.269633,67.35931,0.289514,0.238734,0.471753,0.05078,0.271,0.042,0.687,0.9995,0.970943,0.923676,0.812249,0.70691,0.598333,6.447535,0.196629,0.383436,27.458489,0.893213,24.350555,2.953123,0.0,0.570945,0.005615,0.655089,0.00167,0.606616,0.001604,0.568943,0.007,0.564707,0.007992,0.574247,0.009453,0.563378,0.008194,0.56064,0.00626,0.569879,0.009317,0.566218,0.009741,0.568056,0.011261,0.560285,0.005983,75,10,0.096032,8.253333,2.503541,26.730803,0.479457,0.009592,0.436764,0.003489,0.410795,0.010153,-0.000922,477.405112,4.965306e+06,31811.575437,2.089030e+10,3538.853641,5.815573e+08,1010.554747,3.258910e+07,0,20.177178,0.345174,20.419207
